## 1. Import thư viện

In [1]:
import re
import math
import pickle
import numpy as np
import pandas as pd

from tqdm import tqdm
from collections import Counter, defaultdict
from underthesea import word_tokenize

## 2. Đọc dữ liệu đã xử lý

In [2]:
df = pd.read_pickle("../data/processed/news_processed.pkl")

print("Shape:", df.shape)
df[["id", "title", "topic", "processed_text"]].head()

Shape: (182566, 12)


,id,title,topic,processed_text
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,tên cướp tiệm vàng huế đại_úy công_an công_tác...
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,bỏ_qua mạng 5 g nga tiến thẳng 4 g lên 6 g bỏ_...
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,địa_phương nào đứng đầu cả nước tổng_điểm 3 mô...
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,người chết mưa_lũ nghìn mỹ tăng lên 28 người c...
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...


## 3. Tạo corpus

In [3]:
corpus = df["processed_text"].tolist()

tokenized_corpus = [
    doc.split() for doc in corpus
]

N = len(tokenized_corpus)

print("Number of documents:", N)

Number of documents: 182566


## 4. Tạo vocabulary

In [4]:
vocab = sorted(set(token for doc in tokenized_corpus for token in doc))

print("Vocabulary size:", len(vocab))
print(vocab[:30])

Vocabulary size: 518406
['0', '00', '000', '0000', '0000000000000193', '000008599', '00001', '000012', '000016', '00002', '00005', '0000684894813', '00009', '0001', '00015', '000159', '0002', '0003', '00034', '0005', '00052', '00057', '0006', '00076', '00080', '001', '0010', '0010807', '0011', '0011003814022']


## 5. Tính Document Frequency và IDF

In [5]:
df_count = defaultdict(int)

for doc in tokenized_corpus:
    unique_tokens = set(doc)
    for token in unique_tokens:
        df_count[token] += 1

idf = {}

for word in vocab:
    idf[word] = math.log((N + 1) / (df_count[word] + 1)) + 1

list(idf.items())[:10]

[('0', 3.385797868108875),
 ('00', 6.469781538413446),
 ('000', 2.7469593544512874),
 ('0000', 9.451310861789443),
 ('0000000000000193', 12.421725327359145),
 ('000008599', 12.421725327359145),
 ('00001', 12.016260219250979),
 ('000012', 12.421725327359145),
 ('000016', 12.421725327359145),
 ('00002', 12.421725327359145)]

## 6. Tạo vector TF-IDF cho từng document

In [6]:
tfidf_docs = []

for doc_tokens in tqdm(tokenized_corpus):
    token_counts = Counter(doc_tokens)
    total_tokens = len(doc_tokens)

    doc_tfidf = {}

    if total_tokens > 0:
        for token, count in token_counts.items():
            tf = count / total_tokens
            doc_tfidf[token] = tf * idf.get(token, 0)

    tfidf_docs.append(doc_tfidf)

print("Number of TF-IDF document vectors:", len(tfidf_docs))

100%|██████████| 182566/182566 [00:22<00:00, 7960.21it/s]

Number of TF-IDF document vectors: 182566


## 7. Tạo hàm tiền xử lý query

In [7]:
vietnamese_stopwords = set([
    "là", "và", "của", "có", "cho", "với", "trong", "khi",
    "những", "các", "một", "được", "đã", "đang", "này",
    "đó", "thì", "mà", "ở", "về", "từ", "đến", "theo",
    "sau", "trước", "trên", "dưới", "vào", "ra", "năm",
    "ngày", "tháng", "cũng", "như", "nếu", "vì", "do",
    "để", "bị", "tại", "nên", "sẽ", "rằng", "nhiều"
])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ỹ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_stopwords(text):
    tokens = text.split()
    tokens = [token for token in tokens if token not in vietnamese_stopwords]
    return " ".join(tokens)

def preprocess(text):
    text = clean_text(text)
    text = word_tokenize(text, format="text")
    text = remove_stopwords(text)
    return text

## 8. Vector hóa query theo TF-IDF

In [8]:
def get_query_tfidf(query):
    processed_query = preprocess(query)
    query_tokens = processed_query.split()

    token_counts = Counter(query_tokens)
    total_tokens = len(query_tokens)

    query_tfidf = {}

    if total_tokens > 0:
        for token, count in token_counts.items():
            if token in idf:
                tf = count / total_tokens
                query_tfidf[token] = tf * idf[token]

    return query_tfidf

## 9. Tính cosine similarity thủ công

In [9]:
def cosine_dict(vec1, vec2):
    common_terms = set(vec1.keys()) & set(vec2.keys())

    numerator = sum(vec1[t] * vec2[t] for t in common_terms)

    norm1 = math.sqrt(sum(v ** 2 for v in vec1.values()))
    norm2 = math.sqrt(sum(v ** 2 for v in vec2.values()))

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return numerator / (norm1 * norm2)

## 10. Xây dựng hàm search TF-IDF

In [10]:
def get_tfidf_scores(query):
    query_vec = get_query_tfidf(query)

    scores = []

    for doc_vec in tfidf_docs:
        score = cosine_dict(query_vec, doc_vec)
        scores.append(score)

    return np.array(scores)


def search_tfidf(query, top_k=10):
    scores = get_tfidf_scores(query)

    top_indices = scores.argsort()[::-1][:top_k]

    results = df.iloc[top_indices].copy()
    results["tfidf_score"] = scores[top_indices]

    return results[[
        "id", "title", "topic", "source", "url", "tfidf_score"
    ]]

In [ ]:
## 11. Test TF-IDF Search

In [11]:
search_tfidf("giá xăng tăng", top_k=10)

,id,title,topic,source,url,tfidf_score
56196,148806,Giá xăng tăng bao nhiêu lần từ đầu năm?,Kinh Doanh,vietnamnet.vn,https://vietnamnet.vn/gia-xang-tang-bao-nhieu-...,0.729083
171067,12821,Giá xăng lại chuẩn bị tăng tiếp trong tháng 6?,None,danviet,https://danviet.vn/gia-xang-lai-chuan-bi-tang-...,0.724332
44328,163440,Giá xăng tại Mỹ giảm | VTV.VN,Kinh tế,vtv.vn,https://vtv.vn/kinh-te/gia-xang-tai-my-giam-20...,0.686624
176767,6340,Người Việt ở Mỹ thắt chặt chi tiêu khi giá xăn...,Thế giới,vnexpress,https://vnexpress.net/nguoi-viet-o-my-that-cha...,0.677436
174038,9349,"Xăng tăng giá kỷ lục, bạn sẽ sống sao?",Giới trẻ,thanhnien.vn,https://thanhnien.vn/xang-tang-gia-ky-luc-ban-...,0.677098
162668,21755,Ngày mai (21-6): Dự báo giá xăng tiếp tục lập ...,Kinh doanh,anninhthudo,https://www.anninhthudo.vn/ngay-mai-21-6-du-ba...,0.649001
180094,2475,Giá xăng tại Mỹ sắp đạt kỷ lục mới,Kinh doanh,zingnews,https://zingnews.vn/gia-xang-tai-my-sap-dat-ky...,0.648621
39441,169437,"Giá xăng giảm kỷ lục, giá hàng hóa vẫn ""đứng t...",Kinh tế,danviet,https://danviet.vn/gia-xang-giam-ky-luc-gia-ha...,0.638133
122500,68195,"Giá xăng sẽ giảm vào chiều nay, 1-7?",Kinh tế,qdnd.vn,https://www.qdnd.vn/kinh-te/tin-tuc/gia-xang-s...,0.630754
127838,60277,Giá xăng có thể giảm vào ngày mai | VTV.VN,,vtv.vn,https://vtv.vn/kinh-te/gia-xang-co-the-giam-va...,0.630407


## 12. Lưu TF-IDF index

In [12]:
tfidf_index = {
    "idf": idf,
    "tfidf_docs": tfidf_docs
}

with open("../models/tfidf_index.pkl", "wb") as f:
    pickle.dump(tfidf_index, f)

metadata = df[["id", "title", "topic", "source", "url", "processed_text"]]

metadata.to_pickle("../models/metadata.pkl")

print("Saved:")
print("- ../models/tfidf_index.pkl")
print("- ../models/metadata.pkl")

Saved:
- ../models/tfidf_index.pkl
- ../models/metadata.pkl
